In [ ]:
import torchvision
import torch
import torch.nn as nn

from tqdm import tqdm
import matplotlib.pyplot as plt


In [ ]:
device = torch.accelerator.current_accelerator(check_available=True).type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
train_data = torchvision.datasets.MNIST(root='./data', download=True, train=True,
                                        transform=torchvision.transforms.ToTensor())

test_data = torchvision.datasets.MNIST(root='./data', train=False, transform=torchvision.transforms.ToTensor())


In [ ]:
batch_size = 20
train_dataloader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True)
test_dataloader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.activation = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        conv1 = nn.Conv2d(1, 16, 3, padding=1)
        conv2 = nn.Conv2d(16, 32, 3, padding=1)
        conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.convs = nn.ModuleList([conv1, conv2, conv3])
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(576, 10)

    def forward(self, x):
        for conv in self.convs:
            x = self.pool(self.activation(conv(x)))
        x = self.flatten(x)
        x = self.linear(x)
        return x






In [ ]:
model = CNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)


In [ ]:
def train_loop(model, loss_fn, optimizer, train_dataloader):
    model.train()
    for img, label in tqdm(train_dataloader):
        optimizer.zero_grad()
        output = model(img)
        loss = loss_fn(output, label)
        loss.backward()
        optimizer.step()


def test_loop(model, loss_fn, test_dataloader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for img, label in test_dataloader:
            output = model(img)
            test_loss += loss_fn(output, label).item()
            correct += (output.argmax(1) == label).float().sum().item()
        test_loss /= len(test_dataloader)
        correct /= len(test_dataloader.dataset)
        print(f'test loss:{test_loss:.2f}, test accuracy:{correct * 100:.2f}%')

In [ ]:
epochs = 10
for epoch in range(epochs):
    train_loop(model, loss_fn, optimizer, train_dataloader)
    test_loop(model, loss_fn, test_dataloader)


In [ ]:
def add_noise(img, nosie_level):
    noise = torch.rand(img.shape)
    mask = noise < nosie_level
    noise[~mask] = 0
    return img + noise


test_img, test_label = test_data[0]

img_for_model = add_noise(test_img[0], 0.15).unsqueeze(0).unsqueeze(0)

model.eval()
print(model(img_for_model).argmax(1).item())

plt.imshow(img_for_model[0][0])